In [148]:
import route_matrix
import json
import tqdm

In [149]:
coordinates = json.load(open("data/tract-centroids.json"))
pairs = json.load(open("data/origin_pairs.json"))

In [150]:
def deduplicate(l):
    new_l = []
    for e in l:
        if not e in new_l:
            new_l.append(e)
    return new_l


origin_coords = {k: coordinates[k] for k, v in pairs.items()}
destination_coords = {k: [coordinates[vi] for vi in deduplicate(v)] 
 for k,v in pairs.items()}

destination_coords

{'Census Tract 814.02': [[-87.6853728497224, 41.92862970854046],
  [-87.61975201545292, 41.89503904573094],
  [-87.64296872542037, 41.86789230931981],
  [-87.60996759660905, 41.89190829618846],
  [-87.62517035037656, 41.834959971716984],
  [-87.63274641803926, 41.88102686254463]],
 'Census Tract 813': [[-87.64262363419903, 41.87924795024878],
  [-87.60996759660905, 41.89190829618846],
  [-87.62623002479897, 41.89255912578125],
  [-87.62621204437693, 41.89915380624777],
  [-87.6428042439965, 41.88529624776741],
  [-87.62093223737456, 41.88504648048074],
  [-87.637795508798, 41.89325552522301]],
 'Census Tract 208.01': [[-87.6716805484159, 41.87866521261404],
  [-87.70368969259569, 41.986891558552614],
  [-87.90389546005812, 41.98026693652026],
  [-87.68102501626824, 41.906771599037214],
  [-87.62093223737456, 41.88504648048074]],
 'Census Tract 2410': [[-87.61975201545292, 41.89503904573094],
  [-87.70071234755646, 41.86075009643528],
  [-87.61736675901997, 41.87772138115119],
  [-87.67

In [151]:
for i, (o, c) in enumerate(origin_coords.items()):
    if i: continue
    lon, lat = c

    print(lat)

41.89297495363672


In [152]:
def get_num_signals(city, nodes_visited):
    def has_signal(node):
        attrs = city.nodes[node]
        if not 'highway' in attrs:
            return False
        elif attrs['highway'] == "traffic_signals":
            return True
        else:
            return False

    num_signals = sum([has_signal[p] for p in nodes_visited])
    return num_signals

In [153]:
total_pairs = 0

for i, (o, c) in tqdm.tqdm(enumerate(origin_coords.items())):
    destinations = [tuple([l[1], l[0]]) for l in destination_coords[o]]

    total_pairs += len(destinations)

print(f"TOTAL COMBINATIONS: {total_pairs}")

664it [00:00, 522909.85it/s]

TOTAL COMBINATIONS: 3796


In [ ]:
import datetime as dt
from route_matrix import compute_travel_times, load_api_key

import time

results = json.load(open('./data/real_traffic_paths.json', 'r'))

for i, (o, c) in tqdm.tqdm(enumerate(origin_coords.items())):
    print(i, o, c)
    if o in results: continue
    if i == 345: continue

    lon, lat = c

    origin = (lat, lon)
    destinations = [tuple([l[1], l[0]]) for l in destination_coords[o]]
    partial_res = compute_travel_times(origin, destinations,
                     dt.datetime.fromisoformat("2026-11-03T08:00:00-06:00"), load_api_key('DUMMY'),
                     routing_preference="TRAFFIC_UNAWARE")

    #print(origin)
    #print(destinations)

    for i, r in enumerate(partial_res):
        r['destination'] = deduplicate(pairs[o])[i]
        r['coords'] = coordinates[r['destination']]

        partial_res[i] = r

    results[o] = partial_res

    json.dump(results, open('./data/real_traffic_paths.json', 'w'), indent=2)

542it [00:00, 5044.21it/s]

0 Census Tract 814.02 [-87.60892084919668, 41.89297495363672]
1 Census Tract 813 [-87.62075315477495, 41.89833539670486]
2 Census Tract 208.01 [-87.69441487499267, 41.98680817077086]
3 Census Tract 2410 [-87.69453441576263, 41.9079977707552]
4 Census Tract 601 [-87.6666112467348, 41.95753112664067]
5 Census Tract 307.01 [-87.65796586018746, 41.98274528951097]
6 Census Tract 2215 [-87.68995134669431, 41.92164688814176]
7 Census Tract 1408 [-87.70277513254378, 41.96449419255569]
8 Census Tract 3201 [-87.62093223737456, 41.88504648048074]
9 Census Tract 3510 [-87.60597447545257, 41.83478356548413]
10 Census Tract 713 [-87.64145776838603, 41.9217794296563]
11 Census Tract 8307 [-87.65917156245096, 41.96097268177146]
12 Census Tract 619.02 [-87.63570667263477, 41.944927962514555]
13 Census Tract 8365 [-87.60063789311734, 41.828276986172604]
14 Census Tract 810 [-87.63210893191872, 41.90026556910194]
15 Census Tract 3301 [-87.61529452018165, 41.85938992327736]
16 Census Tract 8310 [-87.67699

657it [00:12, 39.00it/s]  

656 Census Tract 6812 [-87.63849758065729, 41.76918931180475]
657 Census Tract 8426 [-87.65169495935287, 41.81621859470305]


658it [00:12, 38.65it/s]

658 Census Tract 4303 [-87.58935357321577, 41.76969115602987]
659 Census Tract 4606 [-87.55951424897667, 41.73563265817309]
660 Census Tract 7106 [-87.66094335965398, 41.74766006721571]
661 Census Tract 6813 [-87.63986215167523, 41.76117338901422]


664it [00:13, 49.30it/s]

662 Census Tract 6709 [-87.65702200541706, 41.78321241503958]
663 Census Tract 3602 [-87.60990515310972, 41.82687574708339]


In [200]:
import plotly.graph_objects as go
import numpy as np

signal_dict = json.load(open('data/num_traffic_signals.json', 'r'))
budget_results = json.load(open('results/paths/budget/commutes-0-1.json'))
results = json.load(open('./data/real_traffic_paths.json', 'r'))

x = []
y = []

for o, r in results.items():
    for d in r:
        dest = d['destination']

        if not o in budget_results: continue
        if not dest in budget_results[o]: continue
        
        raw_result = budget_results[o][dest]
        realworld_result = d['static_duration_s'] 

        unrestricted_path_index = list(raw_result.keys())[0]
        num_signals = signal_dict[o][dest]

        #if num_signals < 10: continue 

        x += [num_signals]
        if not num_signals:
            y += [0]
        else:
            y += [(realworld_result-raw_result[str(unrestricted_path_index)])/num_signals]


rolling_average = [np.mean(np.array(y)[np.array(x) >= t]) for t in range(max(x)+1)] 
x_average = [np.mean(np.array(y)[np.array(x) == t]) for t in range(1,max(x)+1)] 

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y, mode="markers"))
fig.add_trace(go.Scatter(x=[t for t in range(max(x)+1)], y=rolling_average, name="Mu(X >= x)", line_color="#FF0000"))
fig.add_trace(go.Scatter(x=[t for t in range(max(x)+1)], y=x_average, name="Mu(X == x)", line_color="#00FF00"))
fig.show()



mean = np.mean(y)

x = np.array(x)

x = x[x.argsort()[::1]]
y = np.array(y)[x.argsort()[::1]]





print(f"AVERAGE EXTRA TIME PER TRAFFIC LIGHT: {mean}")

AVERAGE EXTRA TIME PER TRAFFIC LIGHT: 81.77941765897259


In [183]:
rolling_average[10]

np.float64(34.235602098312526)

## Computing signals for origin/destination paths

In [87]:
test = {
    "Census Tract 8412": {
    "Census Tract 207.02": {
      "9": 1299.5094453933395,
      "8": 1302.002729223781,
      "7": 1304.4960130542252,
      "6": 1306.9892968846675,
      "5": 1309.4825807151103,
      "4": 1311.9758645455522,
      "3": 1314.469148375995,
      "2": 1316.962432206439,
      "1": 1319.4557160368809,
      "0": 1321.948999867323
    }
    }
}

print(list(test["Census Tract 8412"]["Census Tract 207.02"].keys())[0])

9
